# Lab 5 — Embeddings y búsqueda semántica

**Minería de Datos · Unidad 2 · UPCh 2026A**
**Alumno:** Carlos Rafael Ramos Molina (231232)
**Grupo:** 9-C
**Profesor:** Ramses Camas Nájera

---

## De qué va este lab

En los labs anteriores usamos TF-IDF y BM25 para buscar documentos. Ambos
sistemas cuentan palabras exactas: si la consulta dice "hídrico" y el
documento dice "agua", no hay match aunque hablen de lo mismo. Eso fue la
falla que vimos en el Lab 4 con los clusters.

Aquí cambiamos eso. Usamos embeddings FastText: representaciones vectoriales
de palabras que capturan su significado según el contexto en que aparecen en
millones de textos. El resultado es que palabras parecidas en significado
quedan cerca en el espacio vectorial, aunque se escriban distinto.

El lab tiene dos partes:
- **Parte A:** explorar ese espacio vectorial (vecinos, relaciones, analogías).
- **Parte B:** construir un buscador semántico y compararlo contra TF-IDF y BM25
  usando las métricas del Lab 3.


## 0 · Corpus del Lab 1

Mismo corpus de siempre. También cargamos aquí las funciones TF-IDF y BM25
de los labs anteriores para usarlas en la comparación de la Parte B.


In [1]:
import json, math
import numpy as np
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}

print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])


14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


### Funciones de los labs anteriores

Pegamos TF-IDF, BM25 y las métricas del Lab 3 para tenerlas disponibles en
la comparación final. Son exactamente las mismas funciones — no cambiamos
nada.


In [2]:
# TF-IDF (Lab 2)
def tf(termino, doc_tokens):
    if not doc_tokens:
        return 0.0
    conteo = Counter(doc_tokens)
    return conteo[termino] / len(doc_tokens)

def idf(termino, documentos):
    N = len(documentos)
    df = sum(1 for doc in documentos if termino in doc)
    return math.log(N / (1 + df)) + 1

def construir_indice(documentos):
    vocabulario = set()
    for doc in documentos:
        vocabulario.update(doc)
    idf_dict = {t: idf(t, documentos) for t in vocabulario}
    vectores_doc = []
    for doc in documentos:
        vec = {}
        for t in set(doc):
            vec[t] = tf(t, doc) * idf_dict[t]
        vectores_doc.append(vec)
    return idf_dict, vectores_doc

def coseno(v1, v2):
    comunes = set(v1.keys()) & set(v2.keys())
    dot = sum(v1[t] * v2[t] for t in comunes)
    n1 = math.sqrt(sum(x**2 for x in v1.values()))
    n2 = math.sqrt(sum(x**2 for x in v2.values()))
    if n1 == 0 or n2 == 0:
        return 0.0
    return dot / (n1 * n2)

def vectorizar_consulta(tokens, idf_dict):
    vec = {}
    for t in set(tokens):
        if t in idf_dict:
            vec[t] = tf(t, tokens) * idf_dict[t]
    return vec

IDF, VECTORES_DOC = construir_indice(documentos)

# BM25 (Lab 3)
avgdl = sum(len(doc) for doc in documentos) / len(documentos)

def idf_bm25(corpus):
    N = len(corpus)
    vocabulario = set()
    for doc in corpus:
        vocabulario.update(doc)
    df_dict = {t: sum(1 for doc in corpus if t in doc) for t in vocabulario}
    return {t: math.log(1 + (N - df + 0.5) / (df + 0.5)) for t, df in df_dict.items()}

IDF_BM25 = idf_bm25(documentos)

def bm25(doc, q, k1=1.5, b=0.75):
    score = 0.0
    doc_len = len(doc)
    conteo_doc = Counter(doc)
    for termino in set(q):
        if termino not in IDF_BM25:
            continue
        f = conteo_doc[termino]
        if f == 0:
            continue
        idf_t = IDF_BM25[termino]
        num = f * (k1 + 1)
        den = f + k1 * (1 - b + b * (doc_len / avgdl))
        score += idf_t * (num / den)
    return score

print('Indices TF-IDF y BM25 listos.')


Indices TF-IDF y BM25 listos.


In [3]:
import re, unicodedata

def preprocesar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return texto.split()

def buscar_tfidf(consulta, k=5):
    tokens = preprocesar(consulta)
    vec_q = vectorizar_consulta(tokens, IDF)
    resultados = [(ids[i], coseno(vec_q, VECTORES_DOC[i])) for i in range(len(ids))]
    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:k]

def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    tokens = preprocesar(consulta)
    resultados = [(ids[i], bm25(documentos[i], tokens, k1, b)) for i in range(len(ids))]
    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:k]

print('Buscadores TF-IDF y BM25 listos.')


Buscadores TF-IDF y BM25 listos.


In [4]:
# Metricas del Lab 3
qrels = {
    'sequia agua escasez': {'d02': 3, 'd04': 3, 'd13': 1},
    'cafe y cacao produccion': {'d03': 3, 'd08': 3, 'd12': 2},
    'turismo y destinos en chiapas': {'d05': 3, 'd09': 3, 'd12': 1},
    'infraestructura y obra publica': {'d10': 3, 'd13': 2, 'd01': 1},
    'tecnologia e innovacion estudiantil': {'d07': 3, 'd14': 3},
}

def _rel(qid, doc):
    return qrels[qid].get(doc, 0)

def _relevantes(qid):
    return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    top = ranking[:k]
    if not top:
        return 0.0
    return sum(1 for d in top if _rel(qid, d) > 0) / len(top)

def recall_at_k(ranking, qid, k=5):
    relevantes = _relevantes(qid)
    if not relevantes:
        return 0.0
    return len(set(ranking[:k]) & relevantes) / len(relevantes)

def mrr(ranking, qid):
    for pos, doc in enumerate(ranking, 1):
        if _rel(qid, doc) > 0:
            return 1.0 / pos
    return 0.0

def average_precision(ranking, qid):
    relevantes = _relevantes(qid)
    if not relevantes:
        return 0.0
    aciertos, suma = 0, 0.0
    for pos, doc in enumerate(ranking, 1):
        if _rel(qid, doc) > 0:
            aciertos += 1
            suma += aciertos / pos
    return suma / len(relevantes) if aciertos > 0 else 0.0

def ndcg_at_k(ranking, qid, k=5):
    top = ranking[:k]
    dcg = sum(_rel(qid, doc) / math.log2(pos + 1) for pos, doc in enumerate(top, 1))
    ideales = sorted(qrels[qid].values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(pos + 1) for pos, rel in enumerate(ideales, 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluar(buscar_fn, k=5, **kwargs):
    acum = {'precision_at_k': [], 'recall_at_k': [], 'mrr': [],
            'average_precision': [], 'ndcg_at_k': []}
    for consulta in qrels:
        res = buscar_fn(consulta, k=max(10, k), **kwargs)
        ranking = [d for d, _ in res]
        acum['precision_at_k'].append(precision_at_k(ranking, consulta, k))
        acum['recall_at_k'].append(recall_at_k(ranking, consulta, k))
        acum['mrr'].append(mrr(ranking, consulta))
        acum['average_precision'].append(average_precision(ranking, consulta))
        acum['ndcg_at_k'].append(ndcg_at_k(ranking, consulta, k))
    return {m: sum(v) / len(v) for m, v in acum.items()}

print('Metricas del Lab 3 listas.')


Metricas del Lab 3 listas.


---

## Parte A · Explorar embeddings en español

### A.1 · Cargar FastText

FastText es un modelo de Facebook que aprendió vectores de palabras a partir
de miles de millones de palabras en texto en español. Lo que lo diferencia de
otros modelos es que maneja morfología: si nunca vio la palabra "chiapaneco",
puede construir su vector a partir de los n-gramas de caracteres que sí conoce
("chiapa", "apas", "panec", etc.). Por eso funciona bien con el español y sus
variaciones.

La descarga (~7 GB) tarda según la conexión — solo la primera vez. Después
se carga del archivo local.

> **Antes de correr esta celda:** cierra Chrome, otras apps pesadas y VS Code
> extensions innecesarias. El modelo ocupa ~7 GB en RAM al cargarse.


In [5]:
import fasttext
import fasttext.util

# la primera vez descarga ~7 GB; las siguientes carga directo del .bin local
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    return ft.get_word_vector(palabra)

print(f'Dimension del embedding : {ft.get_dimension()}')
print(f'Palabras en vocabulario : {len(ft.get_words()):,}')


 (100.00%) [==================================================>]                                                 ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]> 

### A.2 · Vecinos más cercanos

En general los vecinos tienen sentido: para "sequía" salieron variantes de la misma palabra (con/sin tilde, plural) y "inundación", que es un tema relacionado. Para "café" la mayoría son variantes ortográficas de la misma palabra ("café", "cafes", "cafe."), lo cual tiene sentido porque el modelo ve mucho texto con errores de tildes o puntuación pegada. Lo que más me sorprendió fue "chiapas": sus vecinos no son palabras relacionadas en significado, sino otros estados de México (Oaxaca, Michoacán, Veracruz) y la variante "chiapas." con punto. Esto muestra que el modelo aprendió que "Chiapas" pertenece a la categoría "nombres de estados mexicanos" simplemente porque aparecen en contextos parecidos, no porque tengan relación de significado entre sí — es una asociación más bien geográfica/categorial que semántica en el sentido estricto.


In [6]:
palabras = ['sequia', 'cafe', 'chiapas']

for palabra in palabras:
    vecinos = ft.get_nearest_neighbors(palabra, k=5)
    print(f'\nVecinos de "{palabra}":')
    for similitud, vecino in vecinos:
        print(f'  {vecino:<20} {similitud:.4f}')



Vecinos de "sequia":
  sequía               0.7464
  sequias              0.7236
  inundacion           0.5896
  escacez              0.5854
  sequías              0.5713

Vecinos de "cafe":
  café                 0.7897
  cafes                0.7414
  cafe.                0.7375
  cafe-                0.7242
  cafesito             0.7142

Vecinos de "chiapas":
  chiapas.             0.7302
  oaxaca               0.7262
  tuxtla               0.7059
  michoacan            0.6912
  veracruz             0.6861


**Observación (completa con tus resultados):**

Anota si algún vecino te sorprendió, ya sea porque no esperabas verlo ahí
o porque la similitud fue más alta o más baja de lo que intuías.


### A.3 · La falla del agua — ahora a nivel de palabra

En el Lab 4 vimos que TF-IDF no agrupó juntos a d02 ("crisis hídrica") y
d04 ("sequía y maíz") porque usan vocabulario distinto para hablar del mismo
tema. Ahora comprobamos si los embeddings sí ven esa relación: el coseno
entre "agua" e "hídrico" debería ser alto (son conceptualmente cercanos),
y el coseno entre "agua" y "jirafa" debería ser bajo.


In [7]:
def cos_vec(a, b):
    dot = np.dot(a, b)
    return float(dot / (np.linalg.norm(a) * np.linalg.norm(b)))

pares = [
    ('agua', 'hidrico'),
    ('agua', 'sequia'),
    ('agua', 'jirafa'),
    ('cafe', 'cacao'),
    ('cafe', 'sismo'),
]

print(f'{"Par":<30} {"Coseno":>8}')
print('-' * 40)
for a, b in pares:
    sim = cos_vec(vec(a), vec(b))
    print(f'{a} <-> {b:<20} {sim:>8.4f}')


Par                              Coseno
----------------------------------------
agua <-> hidrico                0.4360
agua <-> sequia                 0.3943
agua <-> jirafa                 0.2433
cafe <-> cacao                  0.4511
cafe <-> sismo                  0.0890


**¿Qué demuestra este resultado sobre la falla de los labs anteriores?**

Mi respuesta:

TF-IDF y BM25 solo comparan palabras exactas: si la consulta dice "hídrico"
y el documento dice "agua", no hay ningún match aunque claramente hablen de
lo mismo. Los resultados de coseno entre vectores FastText muestran lo
contrario: "agua" e "hídrico" tienen una similitud alta porque el modelo los
vio aparecer en contextos parecidos durante el entrenamiento (noticias de
abastecimiento, sequía, recursos naturales). Eso es exactamente lo que
le faltaba al clustering del Lab 4, un modelo que entienda que "agua",
"hídrico", "potable" y "escasez" son parte del mismo campo semántico, aunque
no compartan ni una letra.


### A.4 · Analogías por aritmética vectorial

En mi ejecución, la analogía rey-hombre+mujer funcionó perfecto: dio "reina" como primer resultado con 0.6996 de similitud, seguido de "princesa" y "monarca" — todo coherente con el campo de la realeza. La analogía de capitales (México-mexicano+guatemalteco) también funcionó bien, dando "guatemala" como primer resultado con 0.7303, y el resto de los vecinos (tegucigalpa, nicaragua, centroamerica) son todos de la misma región centroamericana, así que el modelo sí entendió la relación "país-gentilicio".
Donde se nota la diferencia es en la tercera analogía (chiapas-mexico+guatemala), que buscaba algo así como "el equivalente de Chiapas pero en Guatemala". Ahí el resultado ya no fue tan limpio: el top no fue un estado o departamento claro de Guatemala, sino "guatemala." (con punto, una variante ortográfica) y "chimaltenango" (que sí es un departamento real de Guatemala) mezclados con "tegucigalpa" (que ni siquiera es de Guatemala, es la capital de Honduras). Esto confirma lo que esperaba: la analogía rey/reina y país/gentilicio funcionan bien porque son relaciones muy frecuentes y consistentes en el texto de entrenamiento, pero una relación más específica como "estado mexicano equivalente a un departamento guatemalteco" es rara en el corpus de FastText, así que el modelo no logra aislar bien esa relación y termina mezclando resultados de toda la región centroamericana en general.


In [8]:
print('Analogia: rey - hombre + mujer')
res = ft.get_analogies('rey', 'hombre', 'mujer', k=5)
for sim, palabra in res:
    print(f'  {palabra:<20} {sim:.4f}')

print('\nAnalogias de capitales: Mexico - mexicano + guatemalteco')
res2 = ft.get_analogies('mexico', 'mexicano', 'guatemalteco', k=5)
for sim, palabra in res2:
    print(f'  {palabra:<20} {sim:.4f}')

print('\nAnalogias: chiapas - mexico + guatemala')
res3 = ft.get_analogies('chiapas', 'mexico', 'guatemala', k=5)
for sim, palabra in res3:
    print(f'  {palabra:<20} {sim:.4f}')


Analogia: rey - hombre + mujer
  reina                0.6996
  princesa             0.6584
  reina-madre          0.5786
  monarca              0.5746
  emperatriz           0.5572

Analogias de capitales: Mexico - mexicano + guatemalteco
  guatemala            0.7303
  tegucigalpa          0.6076
  Guatemala            0.5970
  centroamerica        0.5929
  nicaragua            0.5927

Analogias: chiapas - mexico + guatemala
  guatemala.           0.6077
  tegucigalpa          0.5597
  chimaltenango        0.5537
  Guatemala.           0.5526
  chiapas.             0.5513


**¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?**

Mi respuesta:

Las analogías funcionan mejor cuando las relaciones son consistentes y muy
frecuentes en el corpus de entrenamiento (rey/reina, capital/país), porque
el modelo vio esos pares miles de veces y aprendió el vector que los conecta.
Fallan cuando la relación es más específica o menos frecuente en texto en
español, como topónimos regionales o términos técnicos que aparecen poco. En
esos casos el modelo no tuvo suficientes ejemplos para generalizar bien el
patrón, así que el resultado puede ser una palabra relacionada pero no la
esperada, o incluso algo sin sentido aparente. Completar con lo que
realmente salió en mi ejecución.


---

## Parte B · Buscador semántico sobre el corpus

### B.1 · Vector de documento

La forma más simple de representar un documento con embeddings es promediar
los vectores de sus tokens. Así pasamos de "lista de palabras" a "un solo
vector de 300 dimensiones" que resume el tema del documento.

Tiene limitaciones (no considera el orden de las palabras, ni cuáles son más
importantes), pero para un corpus pequeño funciona razonablemente bien y es
una base sólida antes de usar modelos más sofisticados como Sentence-BERT.


In [9]:
def vector_documento(tokens):
    if not tokens:
        return np.zeros(ft.get_dimension())
    vectores = [vec(t) for t in tokens]
    return np.mean(vectores, axis=0)

# construir un vector por documento
EMB_DOCS = {ids[i]: vector_documento(documentos[i]) for i in range(len(ids))}

print(f'Vectores de documento construidos: {len(EMB_DOCS)}')
print(f'Dimension de cada vector: {list(EMB_DOCS.values())[0].shape}')


Vectores de documento construidos: 14
Dimension de cada vector: (300,)


### B.2 · Buscador semántico

Mismo flujo que TF-IDF y BM25, pero en vez de score de keywords usamos
coseno entre el vector de la consulta y los vectores de los documentos.


In [10]:
def buscar_semantico(consulta, k=5):
    tokens = preprocesar(consulta)
    vec_q = vector_documento(tokens)
    resultados = [(doc_id, cos_vec(vec_q, EMB_DOCS[doc_id])) for doc_id in ids]
    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:k]

# prueba dirigida del enunciado
print('buscar_semantico("problemas de agua"):')
for doc_id, score in buscar_semantico('problemas de agua'):
    print(f'  {doc_id} ({titulos[doc_id]}) — {score:.4f}')


buscar_semantico("problemas de agua"):
  d13 (Restablecen servicio de agua potable) — 0.4584
  d02 (Crisis hidrica golpea la region) — 0.4355
  d01 (Lluvias provocan inundaciones en Tuxtla) — 0.3527
  d10 (Avanza obra de infraestructura carretera) — 0.3384
  d04 (Sequia afecta cultivos de maiz) — 0.3143


### B.3 · Comparación de los tres sistemas

Para 3 consultas, mostramos el top-5 de TF-IDF, BM25 y el buscador
semántico lado a lado. Marcamos los casos donde el semántico recupera
documentos que los otros dos no encuentran.


In [11]:
consultas_comp = [
    'sequia agua escasez',
    'cafe y cacao produccion',
    'tecnologia e innovacion estudiantil',
]

for q in consultas_comp:
    top_tfidf = buscar_tfidf(q, k=5)
    top_bm25  = buscar_bm25(q, k=5)
    top_sem   = buscar_semantico(q, k=5)

    docs_tfidf = [d for d, _ in top_tfidf]
    docs_bm25  = [d for d, _ in top_bm25]
    docs_sem   = [d for d, _ in top_sem]

    print(f'\n=== "{q}" ===')
    print(f'{"Rank":<5}{"TF-IDF":<22}{"BM25":<22}{"Semantico":<22}')
    for i in range(5):
        dt, st = top_tfidf[i]
        db, sb = top_bm25[i]
        ds, ss = top_sem[i]
        marca = ' <-- solo semantico' if ds not in docs_tfidf and ds not in docs_bm25 else ''
        print(f'{i+1:<5}{dt}({st:.2f})'.ljust(22) +
              f'{db}({sb:.2f})'.ljust(22) +
              f'{ds}({ss:.2f})'.ljust(22) + marca)

    solo_sem = set(docs_sem) - set(docs_tfidf) - set(docs_bm25)
    if solo_sem:
        print(f'Docs que solo recupera el semantico: {solo_sem}')



=== "sequia agua escasez" ===
Rank TF-IDF                BM25                  Semantico             
1    d13(0.28)        d02(3.80)             d02(0.74)             
2    d02(0.25)        d13(3.24)             d13(0.67)             
3    d04(0.12)        d04(1.80)             d04(0.57)             
4    d01(0.00)        d01(0.00)             d01(0.51)             
5    d03(0.00)        d03(0.00)             d11(0.51)              <-- solo semantico
Docs que solo recupera el semantico: {'d11'}

=== "cafe y cacao produccion" ===
Rank TF-IDF                BM25                  Semantico             
1    d08(0.29)        d08(4.12)             d12(0.53)             
2    d12(0.25)        d12(3.60)             d03(0.48)             
3    d03(0.12)        d03(1.80)             d08(0.47)             
4    d01(0.00)        d01(0.00)             d04(0.46)              <-- solo semantico
5    d02(0.00)        d02(0.00)             d01(0.41)             
Docs que solo recupera el semantico: 

### B.4 · Re-evaluación con métricas del Lab 3

Reutilizamos los `qrels` y la función `evaluar()` que ya teníamos, ahora
aplicados también al buscador semántico. La métrica clave es NDCG@5 porque
usa relevancia graduada (0–3) — la misma razón por la que la elegimos en
el Lab 3.


In [12]:
def buscar_semantico_eval(consulta, k=5, **kwargs):
    return buscar_semantico(consulta, k=k)

res_tfidf = evaluar(buscar_tfidf, k=5)
res_bm25  = evaluar(buscar_bm25, k=5, k1=1.5, b=0.75)
res_sem   = evaluar(buscar_semantico_eval, k=5)

print(f'{"Metrica":<22}{"TF-IDF":>10}{"BM25":>10}{"Semantico":>12}')
print('-' * 56)
for m in res_tfidf:
    print(f'{m:<22}{res_tfidf[m]:>10.4f}{res_bm25[m]:>10.4f}{res_sem[m]:>12.4f}')


Metrica                   TF-IDF      BM25   Semantico
--------------------------------------------------------
precision_at_k            0.3600    0.3600      0.4800
recall_at_k               0.6000    0.6000      0.8667
mrr                       0.6686    0.6686      0.8667
average_precision         0.5758    0.5758      0.8037
ndcg_at_k                 0.5540    0.5814      0.7764


In [13]:
# NDCG por consulta individual para ver donde mejora mas el semantico
print(f'{"Consulta":<40}{"TF-IDF":>8}{"BM25":>8}{"Sem":>8}')
print('-' * 66)
for consulta in qrels:
    r_tfidf = [d for d, _ in buscar_tfidf(consulta, k=10)]
    r_bm25  = [d for d, _ in buscar_bm25(consulta, k=10)]
    r_sem   = [d for d, _ in buscar_semantico(consulta, k=10)]

    n_t = ndcg_at_k(r_tfidf, consulta, k=5)
    n_b = ndcg_at_k(r_bm25, consulta, k=5)
    n_s = ndcg_at_k(r_sem, consulta, k=5)
    mejor = ' <--' if n_s > max(n_t, n_b) else ''
    print(f'{consulta[:38]:<40}{n_t:>8.4f}{n_b:>8.4f}{n_s:>8.4f}{mejor}')


Consulta                                  TF-IDF    BM25     Sem
------------------------------------------------------------------
sequia agua escasez                       0.8146  0.9514  0.9514
cafe y cacao produccion                   0.9778  0.9778  0.9152
turismo y destinos en chiapas             0.2152  0.2152  0.2781 <--
infraestructura y obra publica            0.7625  0.7625  0.8175 <--
tecnologia e innovacion estudiantil       0.0000  0.0000  0.9197 <--


**¿Mejoró el NDCG? ¿En qué tipo de consultas, y por qué?**

Mi respuesta:

Sí, el NDCG mejoró claramente con el buscador semántico, y de forma muy distinta según el tipo de consulta. El promedio general subió de 0.5540 (TF-IDF) y 0.5814 (BM25) a 0.7764 (semántico), pero el dato más interesante está en el detalle por consulta.
El caso más extremo es "tecnología e innovación estudiantil": tanto TF-IDF como BM25 dieron NDCG de 0.0000 — ni siquiera lograron recuperar un solo documento relevante, porque la consulta usa palabras que no aparecen literalmente en los documentos d07 y d14 (el lab usa "laboratorio", "inteligencia artificial", "robótica", no "tecnología" ni "innovación" tal cual). El buscador semántico en cambio llegó a 0.9197 porque entendió que esas palabras pertenecen al mismo campo conceptual aunque no coincidan exactamente.
También mejoró en "infraestructura y obra pública" (de 0.7625 a 0.8175) y en "turismo y destinos en chiapas" (de 0.2152 a 0.2781), aunque ahí la ganancia es más modesta. Es interesante que en "café y cacao producción" el semántico en realidad bajó un poco (de 0.9778 a 0.9152): ahí TF-IDF y BM25 ya funcionaban muy bien porque la consulta y los documentos comparten palabras exactas, así que no había ninguna falla de vocabulario que el semántico pudiera resolver — y al promediar vectores se pierde algo de precisión fina que el conteo exacto de palabras sí capturaba.
En resumen: el semántico gana más donde la consulta y los documentos relevantes usan palabras distintas para el mismo tema (justo la falla que veníamos arrastrando desde el Lab 4), y no aporta o incluso resta un poco cuando la consulta ya comparte vocabulario literal con los documentos relevantes.

---

## Resumen

- Cargamos FastText en español y exploramos el espacio vectorial (vecinos,
  similitudes, analogías).
- Confirmamos que "agua" e "hídrico" son semánticamente cercanos en el
  embedding — la relación que TF-IDF y BM25 no podían ver.
- Construimos un buscador semántico con vector de documento = promedio de
  embeddings de tokens.
- Comparamos los tres sistemas con NDCG@5 sobre los mismos qrels del Lab 3.
